In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/project")


def read_env(path):
    values = {}
    for number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not key or key != key.strip():
            raise ValueError(f"Invalid KEY=value line {number} in {path}")
        values[key] = value
    return values


CONFIG = read_env(PROJECT_DIR / ".colab.env")
R2_CREDENTIALS = read_env(Path("/content/.colab-r2.env"))
for key in ("R2_BUCKET", "R2_ARTIFACT_PREFIX", "EXPECT_GPU"):
    if not CONFIG.get(key):
        raise ValueError(f"Missing {key} in .colab.env")
for key in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"):
    if not R2_CREDENTIALS.get(key):
        raise ValueError(f"Missing {key} in the external R2 credentials file")
if CONFIG["EXPECT_GPU"] not in ("true", "false"):
    raise ValueError("EXPECT_GPU must be true or false")
ARTIFACT_PREFIX = CONFIG["R2_ARTIFACT_PREFIX"].strip("/")
if not ARTIFACT_PREFIX:
    raise ValueError("R2_ARTIFACT_PREFIX must name a folder")
if CONFIG.get("R2_DATA_PREFIX", "").strip("/") != ARTIFACT_PREFIX:
    raise ValueError("R2_DATA_PREFIX must match R2_ARTIFACT_PREFIX")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

gpu_command = shutil.which("nvidia-smi")
gpu = (
    subprocess.run([gpu_command, "-L"], capture_output=True, text=True)
    if gpu_command
    else None
)
if CONFIG["EXPECT_GPU"] == "true" and (gpu is None or gpu.returncode != 0):
    raise RuntimeError("GPU expected but nvidia-smi did not find one")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")
print(f"Working directory: {Path.cwd()}")
print(f"GPU: {gpu.stdout.strip() if gpu and gpu.returncode == 0 else 'none'}")
print(f"R2 bucket: {CONFIG['R2_BUCKET']} | data: {DATA_DIR} | output: {OUTPUT_DIR}")
print(f"Artifact prefix: {ARTIFACT_PREFIX}")
print("R2 credentials: present")

In [ ]:
%pip install -qq --disable-pip-version-check uv boto3

import subprocess

requirements = PROJECT_DIR / "requirements-colab.txt"
subprocess.run(
    [
        "uv",
        "export",
        "--frozen",
        "--no-dev",
        "--extra",
        "experiment",
        "--prune",
        "torch",
        "--prune",
        "numpy",
        "--prune",
        "fsspec",
        "--prune",
        "rich",
        "--prune",
        "colorama",
        "--no-emit-project",
        "--no-hashes",
        "--format",
        "requirements.txt",
        "--output-file",
        str(requirements),
        "--project",
        str(PROJECT_DIR),
    ],
    check=True,
)
%pip install -qq --disable-pip-version-check --requirement {requirements}
%pip install -qq --disable-pip-version-check --no-deps {PROJECT_DIR}

In [ ]:
from jlens_reasoning.environments.colab import source_bundle_sha256

PROJECT_SOURCE_SHA256 = source_bundle_sha256(PROJECT_DIR)
print(f"Project source SHA-256: {PROJECT_SOURCE_SHA256}")

In [ ]:
def r2_client():
    import boto3

    return boto3.client(
        "s3",
        endpoint_url=f"https://{R2_CREDENTIALS['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
        aws_access_key_id=R2_CREDENTIALS["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=R2_CREDENTIALS["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )


def upload_artifacts():
    """Upload files in OUTPUT_DIR under the project's artifact prefix."""
    client = r2_client()
    count = 0
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_symlink():
            raise ValueError(f"Refusing to upload symlink: {path}")
        if path.is_file():
            key = f"{ARTIFACT_PREFIX}/{path.relative_to(OUTPUT_DIR).as_posix()}"
            client.upload_file(str(path), CONFIG["R2_BUCKET"], key)
            count += 1
    print(f"Uploaded {count} file(s) from {OUTPUT_DIR}")

# FLenQA: second-order probe J-gain

Can we change final-hidden-state sensitivity while preserving the probe projection, and does that change generated answers? See [the short plan](README.md).

Start with **development**: 12 validation problems, one randomly selected matched rendering per problem, 500 versus 3,000 nominal tokens. Evaluate all eligible test problems only after saving development settings. This existing test split has prior descriptive analysis, so its intervention results are **exploratory**.

Shared derivatives and controls live in `jlens_reasoning`; `experiment.py` only orchestrates FLenQA conditions. No static lens asset, retraining, or old answer rerun is needed.

In [ ]:
PHASE = "development"  # development or evaluation
# After inspecting development diagnostics, set one tested strength here to
# save frozen_settings.json. Leave None while exploring; never tune on test.
FREEZE_STRENGTH = None

settings = {
    "layer": 19,
    "short_ctx": 500,
    "long_ctx": 3000,
    "seed": 1729,
    "relative_strengths": [0.001, 0.003, 0.01],  # delta norm / clean hidden norm
    "control_seeds": [11, 29, 47],
    "fd_relative_steps": [0.001, 0.003],
    "fd_rtol": 0.05,
    "fd_atol": 1e-4,
    "probe_score_atol": 1e-4,
    "probe_score_rtol": 1e-4,
    "dtype": "float32",
    "attention": "eager",
}
if PHASE not in ("development", "evaluation"):
    raise ValueError("Choose development or evaluation")
if PHASE == "evaluation" and FREEZE_STRENGTH is not None:
    raise ValueError("Settings can only be frozen from development")

In [ ]:
from jlens_reasoning.environments.colab import download_r2_inputs

paths = [
    "assets/models/qwen3.5-4b/",
    "datasets/flenqa/",
    "checkpoints/flenqa-probe-assets/",
]
if PHASE == "evaluation":
    paths.append("runs/flenqa-probe-jgain/development/")
download_r2_inputs(
    client=r2_client(),
    bucket=CONFIG["R2_BUCKET"],
    prefix=ARTIFACT_PREFIX,
    destination=DATA_DIR,
    paths=paths,
)

In [ ]:
import hashlib
import json
from dataclasses import asdict

import jlens
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_from_disk
from tqdm.auto import tqdm

from experiments.flenqa_probe_jgain.experiment import (
    measure_prompt,
    select_pairs,
    summarize_conditions,
)
from experiments.flenqa_probe_jlens.analysis import validate_split
from experiments.flenqa_probe_jlens.constants import PROBE_CONFIG
from experiments.jlens_readout_sanity.constants import MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.environments.colab import initialize_colab
from jlens_reasoning.experiments_utils.artifacts import write_results
from jlens_reasoning.probing import load_probe_checkpoint, probe_input_contract

context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets"
RESULT_DIR = OUTPUT_DIR / "runs/flenqa-probe-jgain" / PHASE
DEVELOPMENT_DIR = context.runs_dir / "flenqa-probe-jgain/development"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
# Invalidate completion before any partial outputs can replace an older run.
write_results(
    RESULT_DIR / "run_metadata.json",
    {
        "status": "running",
        "phase": PHASE,
        "source_sha256": PROJECT_SOURCE_SHA256,
    },
)


def file_sha256(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

## 1. Bind the run to its inputs

Frozen probes retain their chat-v2 contract. Hash the model files, probe assets, normalized dataset rows, and shared source. Evaluation requires a complete development run and unchanged inputs/code. The notebook source hash is recorded separately because changing phase edits the notebook.

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
rows = normalize_rows(
    dataset["eval"] if hasattr(dataset, "keys") else dataset, full=True
)
split = json.loads((ASSET_DIR / "problem_split.json").read_text())
problem_to_split = validate_split(split, rows)
prompts = prepare_prompts(rows)
prompt_by_id = {prompt.prompt_id: prompt for prompt in prompts}
input_hashes = {
    name: file_sha256(ASSET_DIR / name)
    for name in ("probes.pt", "metadata.json", "problem_split.json")
}
input_hashes["normalized_dataset"] = hashlib.sha256(
    json.dumps([asdict(row) for row in rows], sort_keys=True).encode()
).hexdigest()
input_hashes["model_files"] = {
    path.relative_to(MODEL_PATH).as_posix(): file_sha256(path)
    for path in sorted(Path(MODEL_PATH).rglob("*"))
    if path.is_file()
}
implementation_files = [
    *sorted((PROJECT_DIR / "src").rglob("*.py")),
    PROJECT_DIR / "experiments/flenqa_probe_jgain/experiment.py",
    PROJECT_DIR / "experiments/flenqa_probe_jlens/analysis.py",
    PROJECT_DIR / "experiments/flenqa_probe_jlens/constants.py",
    PROJECT_DIR / "uv.lock",
]
implementation_hashes = {
    str(path.relative_to(PROJECT_DIR)): file_sha256(path)
    for path in implementation_files
}

In [ ]:
frozen = None
if PHASE == "evaluation":
    frozen = json.loads((DEVELOPMENT_DIR / "frozen_settings.json").read_text())
    development = json.loads((DEVELOPMENT_DIR / "run_metadata.json").read_text())
    if development["status"] != "complete":
        raise ValueError("Development run is incomplete")
    for name, digest in frozen["development_hashes"].items():
        if file_sha256(DEVELOPMENT_DIR / name) != digest:
            raise ValueError(f"Development artifact changed: {name}")
    if (
        frozen["input_hashes"] != input_hashes
        or frozen["implementation_hashes"] != implementation_hashes
    ):
        raise ValueError("Inputs or implementation changed since settings were frozen")
    settings = frozen["settings"]
display(settings)

## 2. Select matched problems before observing outcomes

Development uses two problems per task/label stratum from validation. Evaluation uses all eligible test problems. Choose one matched padding/dispersion condition per problem using the fixed seed. Coverage includes every problem in the requested partition, including missing pairs. No selection uses answer correctness or probe strength.

In [ ]:
selected, coverage = select_pairs(
    prompts,
    problem_to_split,
    phase=PHASE,
    seed=settings["seed"],
    short_ctx=settings["short_ctx"],
    long_ctx=settings["long_ctx"],
)
selected_ids = sorted(set(selected.short_id) | set(selected.long_id))
selected.to_parquet(RESULT_DIR / "selected_pairs.parquet", index=False)
coverage.to_parquet(RESULT_DIR / "coverage.parquet", index=False)
display(selected[["problem_id", "task", "label", "padding_type", "dispersion"]])
display(coverage.groupby("reason").size().rename("problems"))

## 3. Load the model for higher-order derivatives

Use float32 and eager attention explicitly for this numerical pilot. All conditions, including clean generation, use that same setup. These are new clean measurements, not assumed identical to previous bfloat16 answers. No fast-kernel fallback or smaller cohort is selected automatically. A derivative or memory failure stops the run; diagnose it before scaling.

The first selected prompt is measured alone below. This checks real-model feasibility before the rest of the cohort. Finite differences probe both sensitivity and the derivative of its squared norm along the gain direction.

In [ ]:
torch.set_float32_matmul_precision("highest")
model = (
    transformers.AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        dtype=getattr(torch, settings["dtype"]),
        attn_implementation=settings["attention"],
        local_files_only=True,
    )
    .to(context.device)
    .eval()
)
model.requires_grad_(False)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
checkpoint = load_probe_checkpoint(
    ASSET_DIR / "probes.pt",
    metadata_path=ASSET_DIR / "metadata.json",
    expected_contract=probe_input_contract(tokenizer, config=PROBE_CONFIG),
    model_name=MODEL_NAME,
)
metadata = checkpoint["metadata"]
if metadata["split"] != split:
    raise ValueError("Probe split differs from the preserved problem split")
if (metadata["num_layers"], metadata["hidden_dim"]) != (
    model.config.num_hidden_layers,
    model.config.hidden_size,
):
    raise ValueError("Model dimensions differ from the frozen probe checkpoint")
probe = checkpoint["layers"][settings["layer"]]
blocks = jlens.from_hf(model, tokenizer, force_bos=False).layers
run_metadata = {
    "status": "running",
    "phase": PHASE,
    "settings": settings,
    "source_sha256": PROJECT_SOURCE_SHA256,
    "implementation_hashes": implementation_hashes,
    "input_hashes": input_hashes,
    "model_name": MODEL_NAME,
    "input_contract": probe_input_contract(tokenizer, config=PROBE_CONFIG),
    "generation_config": asdict(PROBE_CONFIG.inference),
    "source_boundary": "raw block output at final wrapped prompt token",
    "target_boundary": "final normalized hidden state at final wrapped prompt token",
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "device": str(context.device),
    "gpu": torch.cuda.get_device_name(),
    "selected_problems": len(selected),
    "selected_prompts": len(selected_ids),
    "evaluation_scope": "Exploratory: existing test split has prior descriptive analysis",
    "frozen_settings_sha256": file_sha256(DEVELOPMENT_DIR / "frozen_settings.json")
    if frozen
    else None,
}
write_results(RESULT_DIR / "run_metadata.json", run_metadata)

## 4. Measure clean, identity, ±gain, amplification, and random controls

Every strength is a fraction of the clean hidden norm. Random directions use three seeds and the same requested norm. The clean content direction stays frozen for all sensitivity measurements. Generation edits only the first prompt-processing call, then continues normally. Zero content/gain produces explicit status rows; missing answers never count as failed answers. Unsupported computations stop rather than drop a problem.

A partial table is saved after each prompt. Only `status: complete` in the run manifest establishes completion. GPU memory and elapsed time are recorded per prompt.

In [ ]:
# The CLI can continue after cell errors; reset gates on every attempt.
first_prompt_passed = False
cohort_complete = False
measure_kwargs = dict(
    layer=settings["layer"],
    blocks=blocks,
    config=PROBE_CONFIG,
    relative_strengths=settings["relative_strengths"],
    control_seeds=settings["control_seeds"],
    fd_relative_steps=settings["fd_relative_steps"] if PHASE == "development" else (),
)
result_rows, diagnostic_rows = measure_prompt(
    model,
    tokenizer,
    prompt_by_id[selected_ids[0]],
    probe,
    **measure_kwargs,
)
pd.DataFrame(result_rows).to_parquet(RESULT_DIR / "conditions.parquet", index=False)
write_results(RESULT_DIR / "derivative_checks.json", diagnostic_rows)
display(pd.DataFrame(diagnostic_rows))
display(
    pd.DataFrame(result_rows)[
        [
            "condition",
            "relative_strength",
            "status",
            "sensitivity",
            "probe_score",
            "correct",
        ]
    ]
)
if diagnostic_rows:
    first_checks = pd.DataFrame(diagnostic_rows)
    s_ok = np.isclose(
        first_checks.fd_sensitivity,
        first_checks.analytic_sensitivity,
        rtol=settings["fd_rtol"],
        atol=settings["fd_atol"],
    )
    g_ok = np.isclose(
        first_checks.fd_gain_derivative,
        first_checks.analytic_gain_derivative,
        rtol=settings["fd_rtol"],
        atol=settings["fd_atol"],
    )
    if not (s_ok & g_ok).any():
        raise ValueError(
            "First-prompt derivative check failed; inspect before running the cohort"
        )
first_prompt_passed = True

In [ ]:
cohort_complete = False
if not first_prompt_passed:
    raise RuntimeError("First-prompt checks must succeed before running the cohort")
for prompt_id in tqdm(selected_ids[1:], desc=f"J-gain {PHASE}"):
    new_rows, checks = measure_prompt(
        model, tokenizer, prompt_by_id[prompt_id], probe, **measure_kwargs
    )
    result_rows.extend(new_rows)
    diagnostic_rows.extend(checks)
    pd.DataFrame(result_rows).to_parquet(RESULT_DIR / "conditions.parquet", index=False)
    write_results(RESULT_DIR / "derivative_checks.json", diagnostic_rows)
results = pd.DataFrame(result_rows)
expected_conditions = 2 + len(settings["relative_strengths"]) * (
    3 + len(settings["control_seeds"])
)
if len(results) != len(selected_ids) * expected_conditions:
    raise ValueError("Incomplete condition coverage")
cohort_complete = True

## 5. Mechanical and behavioral results

Mechanical success requires ±gain to move sensitivity in opposite directions while preserving probe scores, beyond numerical error and random controls. Inspect finite differences and actual versus requested delta norms before reading accuracy.

Report accuracy changes, repair/damage rates, and percentile intervals by resampling underlying problems. Average random seeds within problems. Compare ±gain with both clean and random controls; the paired long-minus-short contrast asks whether effects are larger at long context. Undefined conditions have explicit coverage counts. Wide intervals mean an inconclusive pilot; preserving one probe score is not preservation of all task information.

In [ ]:
if not cohort_complete:
    raise RuntimeError(
        "Complete the cohort before summarizing or marking the run complete"
    )
summary, contrasts = summarize_conditions(results, seed=settings["seed"])
summary.to_parquet(RESULT_DIR / "mechanical_behavior_summary.parquet", index=False)
contrasts.to_parquet(RESULT_DIR / "accuracy_contrasts.parquet", index=False)
status_counts = results.groupby(["condition", "status"]).size().rename("rows")
display(status_counts)
display(summary)
display(contrasts)
checks = pd.DataFrame(diagnostic_rows)
if len(checks):
    checks["sensitivity_matches"] = np.isclose(
        checks.fd_sensitivity,
        checks.analytic_sensitivity,
        rtol=settings["fd_rtol"],
        atol=settings["fd_atol"],
    )
    checks["gain_matches"] = np.isclose(
        checks.fd_gain_derivative,
        checks.analytic_gain_derivative,
        rtol=settings["fd_rtol"],
        atol=settings["fd_atol"],
    )
    display(checks)
run_metadata["status"] = "complete"
run_metadata["result_hashes"] = {
    path.name: file_sha256(path)
    for path in RESULT_DIR.iterdir()
    if path.suffix in (".parquet", ".json")
    and path.name not in ("run_metadata.json", "frozen_settings.json")
}
write_results(RESULT_DIR / "run_metadata.json", run_metadata)

## 6. Freeze development settings only after inspection

Leave `FREEZE_STRENGTH = None` during development. After reviewing mechanical diagnostics, choose one tested strength and rerun this cell with that value. This does not choose a dose by accuracy. Freezing requires a matching finite-difference check for every nondegenerate prompt, preserved probe scores, and positive/negative mean sensitivity changes. It is a technical gate, not proof of superiority to random controls: inspect the displayed control comparisons before choosing.

Upload the resulting files, switch `PHASE` to `evaluation`, and run a fresh session. Evaluation loads the frozen settings and checks their development artifacts, input hashes, and implementation hashes. It does not retune them.

In [ ]:
if FREEZE_STRENGTH is not None:
    if not cohort_complete:
        raise RuntimeError("Complete the cohort before freezing settings")
    if PHASE != "development" or FREEZE_STRENGTH not in settings["relative_strengths"]:
        raise ValueError("Freeze a strength tested in development")
    nondegenerate = set(
        results.loc[
            results.gain_norm.gt(0) & results.clean_content_norm.gt(0), "prompt_id"
        ]
    )
    if checks.empty or not nondegenerate:
        raise ValueError("No finite-difference evidence available")
    passing = set(
        checks.loc[checks.sensitivity_matches & checks.gain_matches, "prompt_id"]
    )
    if not nondegenerate <= passing:
        raise ValueError(
            "Derivative checks have not passed for every nondegenerate prompt"
        )
    dose = results[
        results.relative_strength.eq(FREEZE_STRENGTH)
        & results.condition.isin(["gain_plus", "gain_minus"])
    ]
    if not dose.status.eq("ok").all():
        raise ValueError(
            "Report and resolve undefined gain cases before freezing this pilot"
        )
    if not np.isclose(
        dose.probe_score,
        dose.clean_probe_score,
        atol=settings["probe_score_atol"],
        rtol=settings["probe_score_rtol"],
    ).all():
        raise ValueError("Probe-score preservation failed at the chosen strength")
    changes = (dose.sensitivity - dose.clean_sensitivity).groupby(dose.condition).mean()
    if not (changes["gain_plus"] > 0 and changes["gain_minus"] < 0):
        raise ValueError(
            "The chosen strength does not change sensitivity bidirectionally"
        )
    frozen_settings = {**settings, "relative_strengths": [FREEZE_STRENGTH]}
    write_results(
        RESULT_DIR / "frozen_settings.json",
        {
            "settings": frozen_settings,
            "input_hashes": input_hashes,
            "implementation_hashes": implementation_hashes,
            "development_hashes": {
                name: file_sha256(RESULT_DIR / name)
                for name in (
                    "run_metadata.json",
                    "conditions.parquet",
                    "derivative_checks.json",
                )
            },
        },
    )
    print("Saved frozen development settings; upload before evaluation.")

In [ ]:
upload_artifacts()
print("COLAB_NOTEBOOK_UPLOAD_COMPLETE")